# Agente Funcional ChileAtiende — EP2 ISY0101
Notebook para Google Colab. Ejecuta las celdas en orden numerado.
> **Importante:** En la celda 2 pega tu `OPENAI_API_KEY`.

## Celda 1 — Instalar dependencias

In [ ]:
!pip install langchain==0.2.16 langchain-openai==0.1.23 langchain-community==0.2.16 chromadb==0.5.5 tiktoken openai -q

## Celda 2 — Configurar API Key

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'TU_API_KEY_AQUI'
print('API Key configurada.')

## Celda 3 — Crear Base de Conocimientos

In [ ]:
import os
os.makedirs('data', exist_ok=True)

fichas = (
    '=== PGU (Pension Garantizada Universal) ===\n'
    'Descripcion: Beneficio del Estado pagado por el IPS.\n'
    'Requisitos:\n'
    '- Tener 65 anos o mas.\n'
    '- No integrar el 10% mas rico de la poblacion.\n'
    '- Residencia en Chile 20 anos o 4 de los ultimos 5 anos previos a la solicitud.\n'
    '- Pension base menor a 1.158.355 pesos.\n'
    'Monto: Maximo 214.296 pesos mensuales.\n'
    'Fuente: https://www.chileatiende.gob.cl/fichas/65066-pension-garantizada-universal-pgu\n\n'
    '=== Bono al Trabajo de la Mujer (BTM) ===\n'
    'Descripcion: Beneficio para mejorar ingresos de mujeres trabajadoras.\n'
    'Requisitos:\n'
    '- Tener entre 25 y 59 anos.\n'
    '- Pertenecer al 40% mas vulnerable segun el RSH.\n'
    '- Cotizaciones al dia.\n'
    '- No trabajar en Institucion del Estado.\n'
    'Monto: Variable segun renta, puede ser anual o mensual.\n'
    'Fuente: https://www.chileatiende.gob.cl/fichas/1285-bono-al-trabajo-de-la-mujer\n\n'
    '=== Subsidio de Arriendo ===\n'
    'Descripcion: Beneficio habitacional del MINVU para arrendar vivienda.\n'
    'Requisitos:\n'
    '- Mayor de 18 anos.\n'
    '- No ser propietario de vivienda.\n'
    '- Pertenecer al 40% mas vulnerable segun el RSH.\n'
    'Monto: Hasta 5 UF mensuales segun zona geografica.\n'
    'Fuente: https://www.chileatiende.gob.cl/fichas/11281-subsidio-de-arriendo\n\n'
    '=== Bono de Invierno ===\n'
    'Descripcion: Pago anual mayo-agosto para gastos de calefaccion.\n'
    'Requisitos:\n'
    '- Tener 65 anos o mas al 31 de mayo del ano del pago.\n'
    '- Ser beneficiario de PGU, PBS o APS.\n'
    'Monto: Entre 50.000 y 100.000 pesos en un solo pago.\n'
    'Fuente: https://www.chileatiende.gob.cl/fichas/2213-bono-de-invierno\n'
)

with open('data/fichas_tramites.txt', 'w', encoding='utf-8') as f:
    f.write(fichas)
print('Base de conocimientos creada.')

## Celda 4 — Definir Herramientas del Agente

In [ ]:
from langchain.tools import tool
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# Construir vectorstore
loader = TextLoader('data/fichas_tramites.txt', encoding='utf-8')
docs = loader.load()
splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=60, separator='\n\n')
splits = splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
print(f'{len(splits)} fragmentos indexados.')

@tool
def buscar_tramite(consulta: str) -> str:
    """Busca informacion oficial sobre bonos y tramites del Estado de Chile en la base de conocimientos de ChileAtiende."""
    resultados = vectorstore.similarity_search(consulta, k=2)
    if not resultados:
        return 'No se encontro informacion relevante en la base de datos.'
    return '\n\n---\n\n'.join([doc.page_content for doc in resultados])

@tool
def redactar_resumen(informacion: str) -> str:
    """Redacta un resumen claro, formal y empatico del tramite para el ciudadano. Maximo 5 oraciones."""
    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.2)
    resp = llm.invoke('Resume de forma clara y empatica para un ciudadano (max 5 oraciones):\n' + informacion)
    return resp.content

@tool
def derivar_ejecutivo(motivo: str) -> str:
    """Deriva al ciudadano a atencion humana cuando la consulta supera el alcance del agente."""
    return (
        'Su consulta requiere atencion personalizada (' + motivo + ').\n'
        'Contacte a ChileAtiende:\n'
        'Call Center: 101 (lun-vie 08:00-22:00)\n'
        'Web: https://www.chileatiende.gob.cl'
    )

TOOLS = [buscar_tramite, redactar_resumen, derivar_ejecutivo]
print('Herramientas:', [t.name for t in TOOLS])

## Celda 5 — Construir el Agente con Memoria

In [ ]:
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_PROMPT = (
    'Eres un agente inteligente oficial de ChileAtiende.\n'
    'Reglas: usa buscar_tramite para consultas sobre bonos/tramites, '
    'redactar_resumen para formatear la respuesta, y derivar_ejecutivo '
    'si la consulta supera tu alcance o involucra datos personales.\n'
    'NUNCA inventes informacion. Usa tono formal y empatico.\n'
    'Historial: {chat_history}'
)

llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad'),
])

agent = create_openai_functions_agent(llm=llm, tools=TOOLS, prompt=prompt)
memory = ConversationBufferWindowMemory(k=5, memory_key='chat_history', return_messages=True, input_key='input')
agent_executor = AgentExecutor(
    agent=agent, tools=TOOLS, memory=memory,
    verbose=True, handle_parsing_errors=True, max_iterations=5
)
print('Agente listo!')

## Celda 6 — Demostracion: 3 consultas de prueba

In [ ]:
# CONSULTA 1: Requisitos de la PGU
r1 = agent_executor.invoke({'input': 'Hola, tengo 67 anos. Cuales son los requisitos para la PGU?'})
print('=== RESPUESTA 1 ===')
print(r1['output'])
print()

# CONSULTA 2: Continuidad con memoria (sin repetir contexto)
r2 = agent_executor.invoke({'input': 'Y cuanto dinero me darian cada mes?'})
print('=== RESPUESTA 2 (prueba de memoria) ===')
print(r2['output'])
print()

# CONSULTA 3: Derivacion a ejecutivo humano
r3 = agent_executor.invoke({'input': 'Necesito saber el estado de mi solicitud, mi RUT es 12.345.678-9'})
print('=== RESPUESTA 3 (derivacion a ejecutivo) ===')
print(r3['output'])